# High-frequency surge ranking — Bella clip 17

Ranks contacts by peak 60–150 Hz power relative to their own pre-onset baseline.

**This is not the Bartolomei EI**, despite the function name: there is no timing
term, so a channel recruited late scores the same as one recruited first. The
repo's EI (`v2/server/app/services/ictal.py`) is the one with the timing term.

Self-contained: needs only `mne` and `numpy`.

In [21]:
import re

import mne
import numpy as np

In [22]:
fn = "../../datasets/Bella_SEEG_EDF/DA6465AU_17_20240319072231.edf"
raw = mne.io.read_raw_edf(fn, preload=True)

Extracting EDF parameters from ../../datasets/Bella_SEEG_EDF/DA6465AU_17_20240319072231.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1446999  =      0.000 ...  1446.999 secs...


# Drop non seeg contacts

Auxiliary traces (REF/DC/EKG/UNUSED) would otherwise be ranked as if they were contacts.

In [23]:
_SEEG_CONTACT_RE = re.compile(r"^[A-Za-z]'?\d+$")


def select_contacts(ch_names):
    """Shaft letter, optional prime for the contralateral shaft, contact number."""
    return [n for n in ch_names if _SEEG_CONTACT_RE.match(str(n).strip())]


contacts = select_contacts(raw.ch_names)
if not contacts:
    raise ValueError(
        f"no channel in {fn} is named like an SEEG contact "
        f"(shaft letter + number, e.g. A1 or X'12); first few are {raw.ch_names[:6]}"
    )

dropped = [n for n in raw.ch_names if n not in set(contacts)]
print(f"dropping {len(dropped)}: {dropped}")
raw.pick(contacts)

dropping 19: ['REF1', 'REF2', 'E', 'DC01', 'DC02', 'DC03', 'DC04', 'DC05', 'DC06', 'DC07', 'DC08', 'DC09', 'DC10', 'UNUSED248', 'UNUSED249', 'UNUSED250', 'UNUSED251', 'EKG1', 'EKG2']


<RawEDF | DA6465AU_17_20240319072231.edf, 184 x 1447000 (1447.0 s), ~1.98 GiB, data loaded>

# Prepare data

60 Hz mains (Cleveland Clinic recording), so notch 60/120/180/240 Hz.

In [24]:
raw.notch_filter(np.arange(60, 241, 60), fir_design='firwin')

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 6601 samples (6.601 s)



<RawEDF | DA6465AU_17_20240319072231.edf, 184 x 1447000 (1447.0 s), ~1.98 GiB, data loaded>

No bipolar re-reference. There is no `mne.set_ieeg_reference`/`set_seeg_reference`,
and `set_eeg_reference` has no `"bipolar"` mode — a bipolar montage is built
explicitly with `mne.set_bipolar_reference(raw, anode=[...], cathode=[...])` from
adjacent-contact pairs you list yourself. Left out because the ranking below is
already per-channel against its own baseline.

# Rank channels

In [25]:
def compute_epileptogenicity_index(raw_data, onset_time, duration=10.0, decim=10):
    """Peak 60-150 Hz power after onset over each channel's own 5 s pre-onset mean."""
    raw_ictal = raw_data.copy().crop(tmin=onset_time - 5, tmax=onset_time + duration)
    sfreq = raw_ictal.info["sfreq"]

    # Event samples are absolute, so a cropped Raw starts at first_samp, not 0.
    # At 0 the epoch falls outside the data and is dropped, leaving none.
    events = np.array([[raw_ictal.first_samp, 0, 1]])
    epochs = mne.Epochs(
        raw_ictal, events, event_id=1, tmin=0,
        tmax=5 + duration - 1.0 / sfreq,  # tmax is inclusive; without this the epoch runs 1 sample past the crop
        baseline=None, preload=True,
    )

    freqs = np.arange(60, 150, 5)
    # tfr_multitaper() is superseded by Epochs.compute_tfr() in mne >= 1.9.
    # decim keeps the channels x freqs x times array to a few hundred MB.
    tfr = epochs.compute_tfr(
        method="multitaper", freqs=freqs, n_cycles=freqs / 2,
        return_itc=False, average=True, decim=decim,
    )

    # uV^2, not V^2: mne returns volts, where power is ~1e-10 and the 1e-6 below
    # stops being a guard and becomes the divisor, flattening every ratio under 1.
    hf_power = tfr.data.mean(axis=1) * 1e12
    baseline_end_idx = int(np.searchsorted(tfr.times, 5.0))

    baseline_mean = hf_power[:, :baseline_end_idx].mean(axis=1, keepdims=True)
    power_ratio = hf_power / (baseline_mean + 1e-6)
    max_surge = np.max(power_ratio[:, baseline_end_idx:], axis=1)

    soz_ranking = {ch_name: max_surge[idx] for idx, ch_name in enumerate(tfr.ch_names)}
    return sorted(soz_ranking.items(), key=lambda x: x[1], reverse=True)

Onsets come from the `.LOG` timing map in
`docs/bella_ictal_ei_vs_annotation_discrepancy.md` — this EDF was converted
before nk2edf wrote log events as annotations, so it carries none of its own.

In [26]:
for onset, label in ((105.0, "A LVFA -> broad"), (117.0, "EEG onset")):
    result = compute_epileptogenicity_index(raw, onset_time=onset)
    print(f"\n=== onset {onset}s ({label}) ===")
    for ch, v in result[:15]:
        print(f"  {ch:8s} {v:10.1f}")
    rank, name = next((i + 1, c) for i, (c, _) in enumerate(result) if re.match(r"^A\d", c))
    print(f"  first shaft-A contact at rank {rank} ({name})")

Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 15000 original time points ...
0 bad epochs dropped

=== onset 105.0s (A LVFA -> broad) ===
  A8          28421.8
  T5          27648.4
  M'9         27379.7
  X'10        17760.3
  T3           9993.7
  M'8          9777.9
  T4           8935.2
  S5           5867.7
  G8           5042.5
  X'11         1078.7
  F8            584.9
  G'9           544.8
  P10           512.5
  F10           328.1
  L8            326.0
  first shaft-A contact at rank 1 (A8)
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 15000 original time points ...
0 bad epochs dropped

=== onset 117.0s (EEG onset) ===
  K2           1269.4
  P10          1168.7
  X'3           862.4
  G'2           859.5
  L'3           809.0
  X3            792.5
  K'2           7